# Import e caricamento dati

In [12]:
import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

import os, sys, pickle
import pandas as pd
from nltk.corpus import wordnet
import math
import ast

sys.path.insert(0, os.path.abspath('..'))
from Sorgente.Preprocessing.preprocesing import Preprocessore
from Sorgente.Indicizzazione.indice_invertito import IndiceInvertito
from Sorgente.Recupero.metadati import GestoreMetadati
from Sorgente.Recupero.bm25 import CalcolatoreRanking
from Sorgente.Recupero.bm25 import pulisci_titolo

In [13]:
indice = IndiceInvertito()
indice.carica('../Dati/Elaborati/indice_fever.pkl')

with open('../Notebook/corpus_fever_arricchito.pkl', 'rb') as f:
    corpus_arricchito = pickle.load(f)

preprocessore = Preprocessore()
metadati = GestoreMetadati(corpus_arricchito)
ranking = CalcolatoreRanking(indice, preprocessore, metadati)

df_campione = pd.read_csv('../Notebook/campione_dev_bilanciato.csv')
print(f'Setup completato. Claim: {len(df_campione):,}')

Setup completato. Claim: 1,500


# Esplorazione WordNet

In [14]:
parola = 'film'
synsets = wordnet.synsets(parola)

print(f"SYNSET DI '{parola}':")
print('=' * 55)
for syn in synsets:
    print(f"\n  Synset: {syn.name()}")
    print(f"  Definizione: {syn.definition()}")
    lemmi = [l.name() for l in syn.lemmas()]
    print(f"  Lemmi (sinonimi): {lemmi}")

SYNSET DI 'film':

  Synset: movie.n.01
  Definizione: a form of entertainment that enacts a story by sound and a sequence of images giving the illusion of continuous movement
  Lemmi (sinonimi): ['movie', 'film', 'picture', 'moving_picture', 'moving-picture_show', 'motion_picture', 'motion-picture_show', 'picture_show', 'pic', 'flick']

  Synset: film.n.02
  Definizione: a medium that disseminates moving pictures
  Lemmi (sinonimi): ['film', 'cinema', 'celluloid']

  Synset: film.n.03
  Definizione: photographic material consisting of a base of celluloid covered with a photographic emulsion; used to make negatives or transparencies
  Lemmi (sinonimi): ['film', 'photographic_film']

  Synset: film.n.04
  Definizione: a thin coating or layer
  Lemmi (sinonimi): ['film']

  Synset: film.n.05
  Definizione: a thin sheet of (usually plastic and usually transparent) material used to wrap or cover things
  Lemmi (sinonimi): ['film', 'plastic_film']

  Synset: film.v.01
  Definizione: make a 

In [15]:
termini_test = ['film', 'born', 'award', 'play', 'star']
print('SINONIMI DAL PRIMO SYNSET')
print('=' * 50)
for termine in termini_test:
    synsets = wordnet.synsets(termine)
    if synsets:
        sinonimi = [l.name().replace('_', ' ')for l in synsets[0].lemmas() if l.name() != termine]
        print(f"  '{termine}' → {sinonimi}")
    else:
        print(f"  '{termine}' → (nessun synset)")

SINONIMI DAL PRIMO SYNSET
  'film' → ['movie', 'picture', 'moving picture', 'moving-picture show', 'motion picture', 'motion-picture show', 'picture show', 'pic', 'flick']
  'born' → ['Born', 'Max Born']
  'award' → ['awarding']
  'play' → ['drama', 'dramatic play']
  'star' → []


In [16]:
named_entities = ['Obama', 'Hawaii', 'NBA', 'Netflix', 'Italy']

print('WORDNET SU NAMED ENTITY')
print('=' * 50)
for ne in named_entities:
    synsets = wordnet.synsets(ne)
    print(f"  '{ne}' → {len(synsets)} synset {'generico' if synsets else 'nessuno — corretto'}")
    if synsets:
        print(f"    Primo synset: {synsets[0].definition()[:60]}")

print()
print('→ Le named entity non hanno sinonimi utili.')
print('  L\'espansione va applicata solo a verbi e sostantivi comuni.')

WORDNET SU NAMED ENTITY
  'Obama' → 0 synset nessuno — corretto
  'Hawaii' → 2 synset generico
    Primo synset: a state in the United States in the central Pacific on the H
  'NBA' → 0 synset nessuno — corretto
  'Netflix' → 0 synset nessuno — corretto
  'Italy' → 1 synset generico
    Primo synset: a republic in southern Europe on the Italian Peninsula; was 

→ Le named entity non hanno sinonimi utili.
  L'espansione va applicata solo a verbi e sostantivi comuni.


# Funzione di espansione

In [17]:
def espandi_termine(termine: str, preprocessore: Preprocessore, max_sinonimi:  int = 3) -> list[str]:
    synsets = wordnet.synsets(termine)
    if not synsets:
        return []
    primo_synset = synsets[0]
    sinonimi_stem = []

    for lemma in primo_synset.lemmas():
        testo_lemma = lemma.name().replace('_', ' ')
        token = preprocessore.elabora(testo_lemma)
        if not token:
            continue
        stem = token[0]
        if stem not in sinonimi_stem:
            sinonimi_stem.append(stem)
        if len(sinonimi_stem) >= max_sinonimi:
            break
    return sinonimi_stem

In [18]:
print('TEST ESPANSIONE TERMINE')
print('=' * 50)
for termine in ['film', 'born', 'award', 'director', 'country']:
    espansi = espandi_termine(termine, preprocessore)
    print(f"  '{termine}' → stem: '{preprocessore.elabora(termine)[0] if preprocessore.elabora(termine) else termine}' → sinonimi: {espansi}")

TEST ESPANSIONE TERMINE
  'film' → stem: 'film' → sinonimi: ['movi', 'film', 'pictur']
  'born' → stem: 'born' → sinonimi: ['born', 'max']
  'award' → stem: 'award' → sinonimi: ['award']
  'director' → stem: 'director' → sinonimi: ['director', 'manag']
  'country' → stem: 'countri' → sinonimi: ['state', 'nation', 'countri']


In [19]:
def espandi_query(query: str, preprocessore: Preprocessore, max_sinonimi: int = 3) -> list[str]:
    token_originali = preprocessore.elabora(query)
    if not token_originali:
        return []
    token_espansi = list(token_originali)
    insieme_token = set(token_originali)
    for token in token_originali:
        sinonimi = espandi_termine(token, preprocessore, max_sinonimi)
        for sinonimo in sinonimi:
            if sinonimo not in insieme_token:
                token_espansi.append(sinonimo)
                insieme_token.add(sinonimo)
    return token_espansi

In [20]:
query_test = [
    'Nikolaj Coster-Waldau worked with Fox Broadcasting Company',
    'Barack Obama was born in Hawaii',
    'The film was directed by an American director',
    'The award was given to the best actor'
]

for query in query_test:
    originali = preprocessore.elabora(query)
    espansi = espandi_query(query, preprocessore)
    aggiunti = [t for t in espansi if t not in originali]
    print(f"\nQuery: '{query}'")
    print(f"  Token originali ({len(originali)}): {originali}")
    print(f"  Aggiunti ({len(aggiunti)}):         {aggiunti}")
    print(f"  Totale espanso ({len(espansi)}):    {espansi}")


Query: 'Nikolaj Coster-Waldau worked with Fox Broadcasting Company'
  Token originali (6): ['nikolaj', 'costerwaldau', 'work', 'fox', 'broadcast', 'compani']
  Aggiunti (0):         []
  Totale espanso (6):    ['nikolaj', 'costerwaldau', 'work', 'fox', 'broadcast', 'compani']

Query: 'Barack Obama was born in Hawaii'
  Token originali (4): ['barack', 'obama', 'born', 'hawaii']
  Aggiunti (3):         ['max', 'hawai', 'aloha']
  Totale espanso (7):    ['barack', 'obama', 'born', 'hawaii', 'max', 'hawai', 'aloha']

Query: 'The film was directed by an American director'
  Token originali (4): ['film', 'direct', 'american', 'director']
  Aggiunti (3):         ['movi', 'pictur', 'manag']
  Totale espanso (7):    ['film', 'direct', 'american', 'director', 'movi', 'pictur', 'manag']

Query: 'The award was given to the best actor'
  Token originali (4): ['award', 'given', 'best', 'actor']
  Aggiunti (4):         ['presumpt', 'precondit', 'histrion', 'player']
  Totale espanso (8):    ['award'

# BM25 con query espansa

In [21]:
def recupera_con_espansione(query: str, indice: IndiceInvertito, preprocessore: Preprocessore, metadati: GestoreMetadati, k: int = 10, max_sinonimi: int = 3) -> list[dict]:
    token_espansi = espandi_query(query, preprocessore, max_sinonimi)
    if not token_espansi:
        return []
    avgdl = indice.lunghezza_media_documenti()
    K1, B = 1.5, 0.75
    ALPHA, BETA = 0.3, 0.2
    punteggi = {}
    for termine in token_espansi:
        df = indice.documenti_con_termine(termine)
        if df == 0:
            continue
        N = indice.numero_documenti
        idf = math.log((N - df + 0.5) / (df + 0.5) + 1)
        posting = indice.cerca(termine)
        for id_doc, tf in posting.items():
            lung = indice.lunghezza_documento(id_doc)
            norm = 1 - B + B * (lung / avgdl)
            tf_bm = (tf * (K1 + 1)) / (tf + K1 * norm)
            punteggi[id_doc] = punteggi.get(id_doc, 0.0) + idf * tf_bm
    risultati = []
    for id_doc, score_bm25 in punteggi.items():
        auth = metadati.authority_score(id_doc)
        fresh = metadati.freshness_score(id_doc)
        risultati.append({
            'id_doc': id_doc,
            'titolo': pulisci_titolo(id_doc),
            'punteggio_finale': round(score_bm25 + ALPHA * auth + BETA * fresh, 4)
        })

    risultati.sort(key=lambda x: -x['punteggio_finale'])
    return risultati[:k]

In [22]:
print('CONFRONTO RANKING — senza vs con espansione WordNet')

for query in query_test:
    senza = ranking.recupera_top_k(query, k=5)
    con = recupera_con_espansione(query, indice, preprocessore, metadati, k=5)
    originali = preprocessore.elabora(query)
    espansi = espandi_query(query, preprocessore)
    aggiunti  = [t for t in espansi if t not in originali]

    print(f"\nQuery: '{query}'")
    print(f"  Sinonimi aggiunti: {aggiunti}")
    print(f"  {'RANK':<5} {'SENZA ESPANSIONE':<45} {'CON ESPANSIONE'}")
    print(f"  {'-'*90}")
    for i, (r_senza, r_con) in enumerate(zip(senza, con), 1):
        uguale = '=' if r_senza['id_doc'] == r_con['id_doc'] else '≠'
        print(f"  {i:<5} {r_senza['titolo']:<45} {uguale} {r_con['titolo']}")

CONFRONTO RANKING — senza vs con espansione WordNet

Query: 'Nikolaj Coster-Waldau worked with Fox Broadcasting Company'
  Sinonimi aggiunti: []
  RANK  SENZA ESPANSIONE                              CON ESPANSIONE
  ------------------------------------------------------------------------------------------
  1     New Amsterdam (TV series)                     = New Amsterdam (TV series)
  2     Nikolaj Coster-Waldau                         = Nikolaj Coster-Waldau
  3     A Thousand Times Good Night                   = A Thousand Times Good Night
  4     Fox Broadcasting Company                      = Fox Broadcasting Company
  5     Nightwatch (1994 film)                        = Nightwatch (1994 film)

Query: 'Barack Obama was born in Hawaii'
  Sinonimi aggiunti: ['max', 'hawai', 'aloha']
  RANK  SENZA ESPANSIONE                              CON ESPANSIONE
  ------------------------------------------------------------------------------------------
  1     Presidency of Barack Obama    

# Valutazione quantitativa

In [23]:
def converti_lista(valore):
    try:
        return ast.literal_eval(str(valore))
    except Exception:
        return []

In [24]:
df_campione['url_documenti'] = df_campione['url_documenti'].apply(converti_lista)
K_VALORI = [1, 3, 5, 10]
CLAIM_DA_TESTARE = 200

hits_senza = {k: 0 for k in K_VALORI}
hits_con = {k: 0 for k in K_VALORI}
totale = 0

claim_supports = df_campione[
    df_campione['label'] == 'SUPPORTS'
].head(CLAIM_DA_TESTARE)

print(f'Valutazione Recall@K su {CLAIM_DA_TESTARE} claim SUPPORTS...')
for _, riga in claim_supports.iterrows():
    doc_attesi = riga['url_documenti']
    if not doc_attesi:
        continue

    res_senza = ranking.recupera_top_k(riga['testo_claim'], k=10)
    id_senza = [r['id_doc'] for r in res_senza]

    res_con = recupera_con_espansione(
        riga['testo_claim'], indice, preprocessore, metadati, k=10
    )
    id_con  = [r['id_doc'] for r in res_con]

    for k in K_VALORI:
        if any(d in set(id_senza[:k]) for d in doc_attesi):
            hits_senza[k] += 1
        if any(d in set(id_con[:k]) for d in doc_attesi):
            hits_con[k] += 1
    totale += 1

print(f'\nRESULTATI (su {totale} claim)')
print(f"{'K':<8} {'Senza espansione':>18} {'Con espansione':>16} {'Differenza':>12}")
print('-' * 56)
for k in K_VALORI:
    r_senza = hits_senza[k] / totale * 100
    r_con = hits_con[k]   / totale * 100
    diff = r_con - r_senza
    segno = '+' if diff >= 0 else ''
    print(f"  Recall@{k:<3} {r_senza:>16.1f}%  {r_con:>14.1f}%  {segno}{diff:>9.1f}%")

Valutazione Recall@K su 200 claim SUPPORTS...

RESULTATI (su 200 claim)
K          Senza espansione   Con espansione   Differenza
--------------------------------------------------------
  Recall@1               79.5%            79.5%  +      0.0%
  Recall@3               95.5%            96.0%  +      0.5%
  Recall@5               98.0%            97.5%       -0.5%
  Recall@10              99.0%            99.0%  +      0.0%


In [25]:
print('CASI IN CUI L\'ESPANSIONE CAMBIA IL RISULTATO')

casi_mostrati = 0
for _, riga in claim_supports.head(500).iterrows():
    doc_attesi = riga['url_documenti']
    if not doc_attesi:
        continue

    res_senza = ranking.recupera_top_k(riga['testo_claim'], k=5)
    res_con = recupera_con_espansione(
        riga['testo_claim'], indice, preprocessore, metadati, k=5
    )

    id_senza_top1 = res_senza[0]['id_doc'] if res_senza else None
    id_con_top1 = res_con[0]['id_doc']   if res_con   else None

    if id_senza_top1 != id_con_top1 and casi_mostrati < 5:
        aggiunti = [
            t for t in espandi_query(riga['testo_claim'], preprocessore)
            if t not in preprocessore.elabora(riga['testo_claim'])
        ]
        print(f"\nClaim:     {riga['testo_claim']}")
        print(f"Aggiunti:  {aggiunti}")
        print(f"Top-1 senza: {pulisci_titolo(id_senza_top1) if id_senza_top1 else '-'}")
        print(f"Top-1 con:   {pulisci_titolo(id_con_top1) if id_con_top1 else '-'}")
        print(f"Ground truth: {doc_attesi}")
        casi_mostrati += 1

if casi_mostrati == 0:
    print('Il top-1 non cambia mai nei primi 500 esempi.')
    print('L\'espansione ha effetto marginale su questo corpus.')

CASI IN CUI L'ESPANSIONE CAMBIA IL RISULTATO

Claim:     The Hundred Years' War includes a war.
Aggiunti:  ['twelvemonth', 'yr', 'warfar']
Top-1 senza: Punic Wars
Top-1 con:   Military deception
Ground truth: ["Hundred_Years'_War"]

Claim:     Benjamin Franklin was a person.
Aggiunti:  ['benzoin', 'gum', 'john', 'individu', 'someon']
Top-1 senza: Franklin stove
Top-1 con:   Tom Wilkinson
Ground truth: ['Benjamin_Franklin', 'Founding_Fathers_of_the_United_States']

Claim:     Elizabeth of York was a person.
Aggiunti:  ['hous', 'individu', 'someon']
Top-1 senza: Elizabeth II
Top-1 con:   Wars of the Roses
Ground truth: ['Elizabeth_of_York']

Claim:     A monster is often grotesque.
Aggiunti:  ['frequent', 'oftentim']
Top-1 senza: Monster
Top-1 con:   Polygyny in animals
Ground truth: ['Monster']

Claim:     Japan is in East Asia.
Aggiunti:  ['japanes', 'due', 'eastward']
Top-1 senza: Korean Peninsula
Top-1 con:   Russo-Japanese War
Ground truth: ['Japan']
